In [2]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from pathlib import Path
import shutil, subprocess

In [12]:
def evaluate_binary_mask(pred_mask, gt_mask, ignore_gt_values=(50, 85, 170)):
    # Convert to single channel if needed
    if pred_mask.ndim == 3:
        pred_mask = cv2.cvtColor(pred_mask, cv2.COLOR_BGR2GRAY)
    if gt_mask.ndim == 3:
        gt_mask = cv2.cvtColor(gt_mask, cv2.COLOR_BGR2GRAY)

    # Build validity mask: ignore uncertain GT labels
    valid = np.ones_like(gt_mask, dtype=bool)
    for v in ignore_gt_values:
        valid &= (gt_mask != v)

    # Binarize prediction and GT
    pred_fg = (pred_mask == 255)
    gt_fg = (gt_mask == 255)

    # Apply valid-mask filtering
    pred_fg = pred_fg[valid]
    gt_fg = gt_fg[valid]

    tp = int(np.sum(pred_fg & gt_fg))
    fp = int(np.sum(pred_fg & (~gt_fg)))
    tn = int(np.sum((~pred_fg) & (~gt_fg)))
    fn = int(np.sum((~pred_fg) & gt_fg))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
frames_dir = Path('../lab2/data/pedestrian/input')

def get_frame(i):
    return cv2.imread(str(frames_dir / f'in{i:06d}.jpg'))

gt_path = Path('../lab2/data/pedestrian/groundtruth')

def get_gt(i):
    return cv2.imread(str(gt_path / f'gt{i:06d}.png'))

first_frame = get_frame(1)

x, y, _ = first_frame.shape
n = 60

buf = np.zeros((x, y, n), dtype=np.uint8)

i_n = 0

mean_bg, median_bg, mean_diff_bin, median_diff_bin, mean_results, median_results = (None,) * 6

for i in range(1, 1000):
    frame = get_frame(i)
    if frame is None:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    buf[:, :, i_n] = frame_gray
    i_n = (i_n + 1) % n
    
    # test on frame 350
    if i == 350:
        print(buf.shape)
        mean_bg = np.mean(buf, axis=2).astype(np.uint8)
        median_bg = np.median(buf, axis=2).astype(np.uint8)
        
        mean_diff_bin = cv2.absdiff(frame_gray, mean_bg)
        _, mean_diff_bin = cv2.threshold(mean_diff_bin, 30, 255, cv2.THRESH_BINARY)
        
        median_diff_bin = cv2.absdiff(frame_gray, median_bg)
        _, median_diff_bin = cv2.threshold(median_diff_bin, 30, 255, cv2.THRESH_BINARY)

        mean_results = evaluate_binary_mask(mean_diff_bin, get_gt(350))
        median_results = evaluate_binary_mask(median_diff_bin, get_gt(350))

        f, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(frame_gray, cmap='gray')
        ax[0].set_title('Original Frame (gray)')
        ax[1].imshow(mean_bg, cmap='gray')
        ax[1].set_title('Mean Background')
        ax[2].imshow(median_bg, cmap='gray')
        ax[2].set_title('Median Background')
        plt.show()

        print('Mean mask metrics:', mean_results)
        print('Median mask metrics:', median_results)

(240, 360, 60)


error: OpenCV(4.11.0) /Users/runner/miniforge3/conda-bld/libopencv_1739279480621/work/modules/core/src/arithm.cpp:665: error: (-209:Sizes of input arguments do not match) The operation is neither 'array op array' (where arrays have the same size and the same number of channels), nor 'array op scalar', nor 'scalar op array' in function 'arithm_op'
